<a href="https://colab.research.google.com/github/aldo02032004/naufaldo.github.io/blob/main/Topic_top_author_sentimen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline: Topik → Top Author → Sentimen

1. **Load data**
2. **Buang akun media**
3. **Bersihin teks**: buang tanda RT/reply, emoji jadi kata, buang teks pendek & duplikat.
4. **Tentukan tema**:
   - AI bikin tema + keyword dari `TOPIC`.
   - Post tanpa keyword → `lainnya`.
   - Sisanya labeling oleh AI.
5. **Top author**: ranking akun per tema dari jumlah post, engagement, dan followers. Ambil 10 teratas.
6. **Sentimen**: AI baca post tiap top author → sikap (positif/negatif/netral/kontroversi) + ringkasan.

**Hasil akhir:** `kol_summary_<project>.csv` di Google Drive.

# Install

In [39]:
import os, shutil

REPO_DIR = "/content/GREAT-Tools"
shutil.rmtree(REPO_DIR, ignore_errors=True)
!git clone -q --depth 1 https://github.com/azmkto/GREAT-Tools.git {REPO_DIR}
!pip install -q emoji ftfy openpyxl tqdm

if not os.path.isdir(f"{REPO_DIR}/great"):
    raise RuntimeError("Gagal")
print("Aman, Semua sudah di download")

Aman, Semua sudah di download


# Import

In [40]:
import json, os, re, sys, time
from collections import Counter
from datetime import date
from typing import Literal

import emoji
import pandas as pd
from google.colab import drive, userdata
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel
from tqdm.auto import tqdm

REPO_DIR = "/content/GREAT-Tools"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from great import EXPORT_COLUMNS, clean_for_bert, clean_for_ner, is_media_account

tqdm.pandas()

AI_MODEL = "gemini-3.7-flash"
MAX_OUTPUT_TOKENS = 32768   # jatah mikir + jawaban AI
SLEEP_BETWEEN_CALLS = 7     # jeda biar ga kena limit tier gratis
api_call_count = 0

# max_retries: kena limit per menit / server error -> otomatis coba ulang
client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
                api_key=userdata.get("GOOGLE_API_KEY"), max_retries=5)


def is_quota_error(e):
    msg = str(e).lower()
    return "429" in msg or "resource_exhausted" in msg or "quota" in msg


def ask(prompt, schema, retry=3):
    """Tanya AI, jawabannya langsung jadi objek sesuai schema. Kuota habis langsung stop, gagal terus balikin None."""
    global api_call_count
    for attempt in range(1, retry + 1):
        try:
            api_call_count += 1
            response = client.beta.chat.completions.parse(
                model=AI_MODEL, max_tokens=MAX_OUTPUT_TOKENS, reasoning_effort="low",
                messages=[{"role": "user", "content": prompt}], response_format=schema)
            return response.choices[0].message.parsed
        except Exception as e:
            if is_quota_error(e):
                raise
            print(f"AI error, coba lagi ({attempt}/{retry}): {e}")
            time.sleep(10 * attempt)
    return None

# form

In [41]:
PROJECT = "prabowo_vladivostok"
TOPIC = "Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin"
SOURCES_TEXT = "https://docs.google.com/spreadsheets/d/1rVvruuhS569La9-lm0qNiIAUoQp-DnnF/edit?usp=drive_link&ouid=116825097454650626545&rtpof=true&sd=true"
REGENERATE_TOPIC_CONFIG = True

PROJECT = re.sub(r"[^a-z0-9_]+", "_", PROJECT.strip().lower()).strip("_")
TOPIC = TOPIC.strip()
SOURCES = [s for s in re.split(r"[,\s]+", SOURCES_TEXT) if s]
if not (PROJECT and TOPIC and SOURCES):
    raise ValueError("PROJECT, TOPIC, sama SOURCES_TEXT harus diisi dulu ya.")
print(f"{PROJECT} | {TOPIC} | {len(SOURCES)} file")

prabowo_vladivostok | Kunjungan Presiden Prabowo ke Vladivostok, Rusia, dan pertemuannya dengan Vladimir Putin | 1 file


# Konfigurasi

In [ ]:
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive"

COL_HEADLINE = "Headline"
COL_MENTIONS = "Mentions"
COL_MEDIA = "Media"
COL_AUTHOR_ID = "Author"
COL_FOLLOWERS = "Followers"
COL_RETWEETED = "Retweeted"
COL_FAVOURITED = "Favourited"
EXPECTED_COLS = [c for c in EXPORT_COLUMNS if c != "Location"]

# bobot skor top author (total 1.0)
WEIGHT_POST_COUNT = 0.6
WEIGHT_ENGAGEMENT = 0.25
WEIGHT_FOLLOWERS = 0.15

BATCH_SIZE = 20              # cuitan per request AI
TOP_N_AUTHORS = 10           # per tema
MAX_POSTS_PER_AUTHOR = 15    # post yang dibaca AI untuk sentimen
MIN_THEME_CONFIDENCE = 0.6   # di bawah ini tidak ikut ranking
DEDUP_PER_AUTHOR = False     # True = post kembar dari author berbeda tetap dihitung

TOPIC_CONFIG_PATH = f"{DRIVE_DIR}/topic_config_{PROJECT}.json"
CHECKPOINT_PATH = f"{DRIVE_DIR}/checkpoint_tema_{PROJECT}.pkl"
THEME_RESULT_PATH = f"{DRIVE_DIR}/df_clean_after_theme_{PROJECT}.pkl"
TOP_AUTHORS_PATH = f"{DRIVE_DIR}/kol_{PROJECT}.csv"
SUMMARY_PATH = f"{DRIVE_DIR}/kol_summary_{PROJECT}.csv"
PENDING_TEMA_PATH = f"{DRIVE_DIR}/pending_tema_data.jsonl"   # data latih tema, dipakai semua topik

# Konfigurasi topik (dibuat Colab AI)

In [ ]:
TOPIC_CONFIG_PROMPT = f"""You are a media analyst at a socio-political think tank, setting up monitoring of Indonesian online news and social media.

Topic: "{TOPIC}" (today: {date.today()})

The raw data is broader than this topic (e.g. every post that mentions its main figure). Posts without any keyword are dropped; the rest are classified into themes.

Return JSON with exactly these fields:
1. themes         - 2-4 themes that split the discussion by angle (e.g. event, policy, political reaction), not by person. Each has a short snake_case name and a description: "Masuk: <what it covers>. Contoh frasa: <8-15 typical phrases from news and social media, incl. slang>. Bukan: <similar things that belong to another theme>." Do not create a "lainnya" theme.
2. keywords       - 20-60 lowercase words or short phrases that almost only appear when this topic is discussed: other parties, places, events, institutions, abbreviations, common typos, hashtags without #. Matched as whole words, so prefer single words. Never the main figure's name alone, never general words used in other issues (e.g. "investasi", "kunjungan").
3. media_accounts - lowercase usernames (no @) of news outlets or official institution accounts tied to this topic. Exclude personal accounts, buzzers, and influencers. If unsure, leave it out; an empty list is fine.

Answer in Indonesian. Output only JSON."""


class Theme(BaseModel):
    name: str
    description: str


class TopicConfig(BaseModel):
    themes: list[Theme]
    keywords: list[str]
    media_accounts: list[str]


def check_config(cfg):
    """Rapiin config: nama tema snake_case, keyword huruf kecil, buang yang aneh."""
    themes = {re.sub(r"\W+", "_", k.strip().lower()).strip("_"): v for k, v in cfg["themes"].items()}
    themes.pop("lainnya", None)
    keywords = {k.strip().lower().lstrip("#") for k in cfg["keywords"]}
    keywords = sorted(k for k in keywords if len(k) >= 3)   # kata 1-2 huruf terlalu ambigu
    if not themes or not keywords:
        raise RuntimeError("Tema/keyword dari AI kosong. Jalanin ulang cell ini.")
    media = sorted({a.strip().lower().lstrip("@") for a in cfg.get("media_accounts", [])})
    return {"themes": themes, "keywords": keywords, "media_accounts": media}


topic_config = {}
if os.path.exists(TOPIC_CONFIG_PATH) and not REGENERATE_TOPIC_CONFIG:
    with open(TOPIC_CONFIG_PATH, encoding="utf-8") as f:
        topic_config = json.load(f)

if topic_config.get("_topic") != TOPIC:   # belum ada, minta dibikin ulang, atau punya topik lain
    answer = ask(TOPIC_CONFIG_PROMPT, TopicConfig)
    if answer is None:
        raise RuntimeError("AI gagal bikin konfigurasi topik. Jalanin ulang cell ini.")
    topic_config = {"themes": {t.name: t.description for t in answer.themes},
                    "keywords": answer.keywords, "media_accounts": answer.media_accounts, "_topic": TOPIC}
    with open(TOPIC_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(topic_config, f, ensure_ascii=False, indent=2)

topic_config = check_config(topic_config)
THEMES_DESC = {**topic_config["themes"], "lainnya": "di luar tema di atas"}
THEMES = list(THEMES_DESC)
RELEVANT_THEMES = THEMES[:-1]
KEYWORDS = sorted(topic_config["keywords"], key=len, reverse=True)   # frasa panjang dicocokin duluan
MEDIA_ACCOUNTS = set(topic_config["media_accounts"])
RE_KEYWORD = re.compile(r"(?<!\w)(?:" + "|".join(map(re.escape, KEYWORDS)) + r")(?!\w)")

print(json.dumps(topic_config, ensure_ascii=False, indent=2))

# Step 1 — Load data

In [ ]:
def xlsx_url(link):
    """Link Google Sheets/Drive -> link download .xlsx."""
    match = re.search(r"/d/([\w-]+)", link)
    if not match:
        return link
    if "drive.google.com" in link:
        return f"https://drive.google.com/uc?export=download&id={match.group(1)}"
    return f"https://docs.google.com/spreadsheets/d/{match.group(1)}/export?format=xlsx"


def load_sheet(link):
    try:
        df = pd.read_excel(xlsx_url(link), header=1)
    except ValueError as e:
        raise ValueError(f"Gagal baca {link}. Pastikan akses file 'Anyone with the link'.") from e
    df.columns = df.columns.str.strip()
    return df[EXPECTED_COLS]


df_raw = pd.concat([load_sheet(s).assign(source_file=f"file_{i}") for i, s in enumerate(SOURCES, 1)],
                   ignore_index=True)
print(f"{len(df_raw)} baris")
display(df_raw.head())

# Step 2 — Buang akun media

In [ ]:
NUM_COLS = [COL_FOLLOWERS, COL_RETWEETED, COL_FAVOURITED]
df_raw[NUM_COLS] = df_raw[NUM_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)

is_media = pd.Series([is_media_account(a, m, extra_accounts=MEDIA_ACCOUNTS)
                      for a, m in zip(df_raw[COL_AUTHOR_ID], df_raw[COL_MEDIA])], index=df_raw.index)
print(f"Buang {is_media.sum()} baris media, contoh akun:",
      df_raw.loc[is_media, COL_AUTHOR_ID].astype(str).unique()[:20].tolist())
df_raw = df_raw[~is_media].reset_index(drop=True)

# Step 3 — Bersihkan teks

In [ ]:
# Retweet ditulis "RT <teks>", balasan ditulis "... [RE username]"
RE_RT = re.compile(r"^\s*RT\b\s*(?:@\w+\s*:\s*)?")
RE_REPLY = re.compile(r"\[RE\s+@?\w+\]", re.IGNORECASE)
RE_LONG = re.compile(r"(.)\1{2,}")   # "mantaaap" -> "mantaap"


def strip_noise(text):
    """Buang tanda RT/[RE], emoji jadi kata, huruf yang diulang-ulang dipendekin."""
    text = RE_REPLY.sub(" ", RE_RT.sub("", text))
    text = emoji.demojize(text, language="id").replace("_", " ").replace(":", " ")
    return RE_LONG.sub(r"\1\1", text)


def combine_text(headline, mentions):
    headline = "" if pd.isna(headline) else str(headline).strip()
    mentions = "" if pd.isna(mentions) else str(mentions).strip()
    if headline.lower() in ("", "nan", mentions.lower()):
        return mentions or headline
    return f"{headline}. {mentions}" if mentions else headline


df_raw["text_raw"] = [combine_text(h, m) for h, m in zip(df_raw[COL_HEADLINE], df_raw[COL_MENTIONS])]
df_raw["is_retweet"] = df_raw["text_raw"].str.match(RE_RT)
no_noise = df_raw["text_raw"].map(strip_noise)
df_raw["text_clean"] = no_noise.progress_apply(clean_for_bert)   # buat filter, cek duplikat & keyword
df_raw["text_ai"] = no_noise.apply(clean_for_ner)                # buat AI: huruf besar, slang & @akun tetap

df_clean = df_raw[df_raw["text_clean"].str.split().str.len() >= 3]

# buang duplikat; post asli didahulukan biar yang kesisa punya penulis aslinya
dup_key = df_clean["text_clean"].str.lower().str.replace(r"[^\w\s]", "", regex=True).str.split().str.join(" ")
dup_cols = ["dup_key", COL_AUTHOR_ID] if DEDUP_PER_AUTHOR else ["dup_key"]
is_dup = df_clean.assign(dup_key=dup_key).sort_values("is_retweet", kind="stable").duplicated(dup_cols).sort_index()
df_clean = df_clean[~is_dup].reset_index(drop=True)

print(f"{len(df_raw)} -> {len(df_clean)} baris setelah buang teks pendek & duplikat "
      f"({df_raw['is_retweet'].sum()} retweet ketemu)")
display(df_clean[["text_raw", "text_clean", "text_ai"]].head())

# Step 4 — Tentukan tema (Colab AI)

In [ ]:
THEME_LIST = "\n".join(f"- {t}: {d}" for t, d in THEMES_DESC.items())


def theme_prompt(texts):
    numbered = "\n".join(f'{i}. "{t}"' for i, t in enumerate(texts, 1))
    return f"""You are a media analyst at a socio-political think tank, classifying Indonesian news and social media posts about: {TOPIC}.

Themes:
{THEME_LIST}

Posts:
{numbered}

Return JSON with one entry per post ({len(texts)} posts), each with these fields:
1. index         - the post number.
2. primary_theme - one theme name from the list. Use "lainnya" if the post is not directly about the topic (e.g. it only mentions the same person, country, or institution for another event).
3. confidence    - 0 to 1.

Output only JSON."""


class PostTheme(BaseModel):
    index: int
    primary_theme: Literal[tuple(THEMES)]
    confidence: float


class ThemeAnswer(BaseModel):
    posts: list[PostTheme]


def save_checkpoint(next_batch):
    pd.to_pickle({"run_id": run_id, "df": df_clean, "todo": todo, "next": next_batch, "tema_rows": tema_rows},
                 CHECKPOINT_PATH)


# checkpoint cuma dilanjutin kalau topik, tema, sama datanya masih sama
run_id = [TOPIC, THEMES, int(pd.util.hash_pandas_object(df_clean["text_clean"], index=False).sum())]
ckpt = pd.read_pickle(CHECKPOINT_PATH) if os.path.exists(CHECKPOINT_PATH) else {}

if ckpt.get("run_id") == run_id:
    df_clean, todo, start, tema_rows = ckpt["df"], ckpt["todo"], ckpt["next"], ckpt["tema_rows"]
    print(f"Lanjut dari batch {start}")
else:
    df_clean[["theme", "theme_confidence"]] = None
    # post tanpa keyword langsung "lainnya", ga usah dikirim ke AI
    text_lower = df_clean["text_clean"].str.lower()
    no_kw = ~text_lower.str.contains(RE_KEYWORD)
    df_clean.loc[no_kw, ["theme", "theme_confidence"]] = ["lainnya", 1.0]
    print(f"{no_kw.sum()} post ga ada keyword -> 'lainnya', {(~no_kw).sum()} post dikirim ke AI")
    print("Keyword paling sering:",
          Counter(k for t in text_lower[~no_kw] for k in set(RE_KEYWORD.findall(t))).most_common(10))
    if (~no_kw).mean() > 0.5:
        print("Lebih dari separuh post lolos, kayaknya ada keyword yang terlalu umum (cek di atas). "
              "Centang REGENERATE_TOPIC_CONFIG terus jalanin ulang.")
    todo = df_clean.index[df_clean["theme"].isna()].tolist()
    start, tema_rows = 0, []

n_batches = -(-len(todo) // BATCH_SIZE)
finished = True
for b in tqdm(range(start, n_batches), initial=start, total=n_batches, desc="Tema"):
    idx = todo[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
    texts = df_clean.loc[idx, "text_ai"].tolist()
    try:
        answer = ask(theme_prompt(texts), ThemeAnswer)
    except Exception as e:
        if not is_quota_error(e):
            raise
        save_checkpoint(b)
        print(f"Kuota habis di batch {b}/{n_batches}. Tunggu bentar terus jalanin ulang cell ini.")
        finished = False
        break

    found = {p.index: p for p in answer.posts} if answer else {}
    for n, i in enumerate(idx, 1):
        if n in found:
            p = found[n]
            df_clean.loc[i, ["theme", "theme_confidence"]] = [p.primary_theme, p.confidence]
            tema_rows.append({"text": df_clean.at[i, "text_clean"], "label": p.primary_theme,
                              "confidence": p.confidence})
    if len(found) < len(idx):
        print(f"Batch {b}: {len(idx) - len(found)} post ga dapet tema dari AI.")

    if b % 10 == 9:
        save_checkpoint(b + 1)
    time.sleep(SLEEP_BETWEEN_CALLS)

if finished:
    # data latih ditulis sekali di akhir biar ga dobel waktu lanjut dari checkpoint
    with open(PENDING_TEMA_PATH, "a", encoding="utf-8") as f:
        for r in tema_rows:
            f.write(json.dumps({**r, "label_source": "gemini", "task": "tema", "project": PROJECT,
                                "run_date": str(date.today())}, ensure_ascii=False) + "\n")
    if os.path.exists(CHECKPOINT_PATH):
        os.remove(CHECKPOINT_PATH)

n_missing = df_clean["theme"].isna().sum()
if n_missing:
    print(f"{n_missing} post belum punya tema, jadi hasil di bawah belum lengkap.")
display(df_clean["theme"].value_counts(dropna=False))
df_clean.to_pickle(THEME_RESULT_PATH)
print(f"Disimpen ke {THEME_RESULT_PATH}. Total request AI: {api_call_count}")

# Step 5 — Top author per tema

In [ ]:
relevant = df_clean["theme"].isin(RELEVANT_THEMES)
confident = df_clean["theme_confidence"].astype(float) >= MIN_THEME_CONFIDENCE
# retweet ga dihitung, soalnya bukan tulisan akun itu sendiri
df_posts = df_clean[relevant & confident & ~df_clean["is_retweet"]].copy()
df_posts["engagement"] = df_posts[COL_FAVOURITED] + df_posts[COL_RETWEETED]
print(f"{len(df_posts)} post asli ikut ranking. Dibuang: {(relevant & ~confident).sum()} tema kurang yakin, "
      f"{(relevant & confident & df_clean['is_retweet']).sum()} retweet")

authors = (df_posts.groupby(["theme", COL_AUTHOR_ID])
           .agg(jumlah_post=("engagement", "size"),
                total_engagement=("engagement", "sum"),
                followers=(COL_FOLLOWERS, "max"))
           .reset_index()
           .rename(columns={COL_AUTHOR_ID: "author_id"}))
metrics = authors[["jumlah_post", "total_engagement", "followers"]].astype(float)
# dibagi angka paling gede di temanya, jadi skalanya 0-1
norm = metrics / metrics.groupby(authors["theme"]).transform("max").replace(0, 1)
authors["score"] = norm @ [WEIGHT_POST_COUNT, WEIGHT_ENGAGEMENT, WEIGHT_FOLLOWERS]

top_authors = (authors.sort_values(["theme", "score"], ascending=[True, False])
               .groupby("theme").head(TOP_N_AUTHORS)
               .reset_index(drop=True))
top_authors["rank"] = top_authors.groupby("theme").cumcount() + 1
top_authors.to_csv(TOP_AUTHORS_PATH, index=False)

for theme in RELEVANT_THEMES:
    group = top_authors[top_authors["theme"] == theme]
    print(f"\n=== {theme} ===")
    if group.empty:
        print("ga ada author")
    else:
        display(group[["rank", "author_id", "jumlah_post", "total_engagement", "followers", "score"]])

# Step 6 — Sentimen & ringkasan per author (Colab AI)

In [ ]:
def sentiment_prompt(author_id, theme, posts):
    joined = "\n".join(f"- {p}" for p in posts)
    return f"""You are a media analyst at a socio-political think tank, profiling one account's stance.

These posts by "{author_id}" are about the theme "{theme.replace('_', ' ')}" of the topic "{TOPIC}":
{joined}

Return JSON with exactly these fields:
1. sentiment - the account's stance toward the topic, not its tone: positif (supports, praises, defends), negatif (criticises, rejects, mocks), netral (only informs, no clear stance), or kontroversi (support and criticism equally strong). Defending the main figure while insulting critics is positif.
2. summary   - 2-3 sentences on the account's main view, grounded in the posts.
3. reason    - one sentence explaining the label, grounded in the posts.

Answer in Indonesian. Output only JSON."""


class Sentiment(BaseModel):
    sentiment: Literal["positif", "negatif", "netral", "kontroversi"]
    summary: str
    reason: str


results = []
for _, a in tqdm(top_authors.iterrows(), total=len(top_authors), desc="Sentimen"):
    posts = (df_posts[(df_posts[COL_AUTHOR_ID] == a["author_id"]) & (df_posts["theme"] == a["theme"])]
             .nlargest(MAX_POSTS_PER_AUTHOR, "engagement")["text_ai"].tolist())
    try:
        answer = ask(sentiment_prompt(a["author_id"], a["theme"], posts), Sentiment)
    except Exception as e:
        if not is_quota_error(e):
            raise
        print(f"Kuota habis. {len(results)} author udah dinilai, sisanya jalanin ulang nanti.")
        break
    results.append(answer.model_dump() if answer else {"sentiment": None, "summary": "", "reason": ""})

df_summary = pd.concat([top_authors.head(len(results)),
                        pd.DataFrame(results, columns=["sentiment", "summary", "reason"])], axis=1)
df_summary.to_csv(SUMMARY_PATH, index=False)

n_failed = df_summary["sentiment"].isna().sum()
if n_failed:
    print(f"{n_failed} author gagal dinilai AI (sentiment kosong). Jalanin ulang cell ini.")
print(f"Disimpen ke {SUMMARY_PATH}. Total request AI: {api_call_count}")

pd.set_option("display.max_colwidth", None)
display(df_summary.round({"score": 2}))